In [1]:
import sys
from pathlib import Path
from omegaconf import OmegaConf

In [2]:
base_path = Path('/workspace/src').resolve()
src_path = base_path / 'src'
cfg_path = base_path / 'config'
pmri_data_path = Path('/data/Data/PMRI').resolve()
sys.path.append(str(src_path))
cfg = OmegaConf.load(cfg_path / 'conf.yaml')

In [3]:
DATASET_KEY = 'prostate'
DATASET_SUBKEY = 'pmri'
ARCH = 'monai-unet-64-4-4'
PROJECT_NAME = 'test-UNet'
VALIDATION = True
OmegaConf.update(cfg, 'run.dataset_key', DATASET_KEY)
OmegaConf.update(cfg, 'run.dataset_subkey', DATASET_SUBKEY)
OmegaConf.update(cfg, 'run.arch', ARCH)
OmegaConf.update(cfg, 'run.validation', VALIDATION)
OmegaConf.update(cfg, 'wandb.project', PROJECT_NAME)

In [4]:
from datasets import load_dataset

In [5]:
data = load_dataset(cfg)
ds_train = data['train']
ds_valid = data['valid']

Loading dataset...


KeyboardInterrupt: 

In [ ]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import optim, nn
from monai.networks.nets import DynUNet
from monai.data import DataLoader, Dataset
from monai.transforms import (
    CastToTyped,
    Compose,
    EnsureTyped,
    RandFlipd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandScaleIntensityd,
    RandZoomd,
    SpatialPadd,
    ToTensord,
)

#### Transforms flipd axis test

In [ ]:
tr0 = Compose([
    ToTensord(keys=['image', 'label']),
    RandFlipd(keys=['image', 'label'], spatial_axis=[0], prob=1.0),
])
tr1 = Compose([
    ToTensord(keys=['image', 'label']),
    RandFlipd(keys=['image', 'label'], spatial_axis=[1], prob=1.0),
])

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(8,8))
axs[0].imshow(ds_train[0]['image'].squeeze(), cmap='gray')
axs[0].set_title('Image original')
axs[1].imshow(ds_train[0]['label'].squeeze(), cmap='gray')
axs[1].set_title('Label original')
axs[2].imshow(ds_train[0]['image'].squeeze(), cmap='gray')
axs[2].imshow(ds_train[0]['label'].squeeze(), cmap='gray', alpha=0.5)
axs[2].set_title('Overlaped original')
plt.show();
fig, axs = plt.subplots(1,3, figsize=(8,8))
tmp = tr0(ds_train[0])
axs[0].imshow(tmp['image'].squeeze(), cmap='gray')
axs[0].set_title('Image flip x')
axs[1].imshow(tmp['label'].squeeze(), cmap='gray')
axs[1].set_title('Label flip x')
axs[2].imshow(tmp['image'].squeeze(), cmap='gray')
axs[2].imshow(tmp['label'].squeeze(), cmap='gray', alpha=0.5)
axs[2].set_title('Overlaped flip x')
plt.show();
fig, axs = plt.subplots(1,3, figsize=(8,8))
tmp = tr1(ds_train[0])
axs[0].imshow(tmp['image'].squeeze(), cmap='gray')
axs[0].set_title('Image flip 1 (y)')
axs[1].imshow(tmp['label'].squeeze(), cmap='gray')
axs[1].set_title('Label flip 1 (y)')
axs[2].imshow(tmp['image'].squeeze(), cmap='gray')
axs[2].imshow(tmp['label'].squeeze(), cmap='gray', alpha=0.5)
axs[2].set_title('Overlaped flip y')
plt.show();

#### Actual test UNet

In [ ]:
# transforms = Compose([
#     ToTensord(keys=['image', 'label']),
#     SpatialPadd(keys=['image', 'label'], spatial_size=ds_train._DS_CONFIG['size']),
#     RandZoomd(keys=['image', 'label'], min_zoom=0.9, max_zoom=1.2,
#               mode=('trilinear', 'nearest'), align_corners=(True, None), prob=0.15),
#     RandGaussianNoised(keys=['image'], std=0.01, prob=0.15),
#     RandGaussianSmoothd(keys=['image'], sigma_x=(0.5, 1.15),
#                         sigma_y=(0.5, 1.15), prob=0.15),
#     RandScaleIntensityd(keys=['image'], factors=0.3, prob=0.15),
#     RandFlipd(keys=['image', 'label'], spatial_axis=[0], prob=0.5),
#     RandFlipd(keys=['image', 'label'], spatial_axis=[1], prob=0.5),
#     CastToTyped(keys=['image', 'label'], dtype=(np.float32, torch.long)),
#     EnsureTyped(keys=['image', 'label'])
# ])

In [ ]:
from data_utils import Transform
from trainer import train_loop
from models import get_model

In [ ]:
transforms = Transform(cfg)
model_type = cfg.run.arch.split('-')[1]
batch_size = cfg[model_type][DATASET_KEY].training.batch_size
lr = cfg[model_type][DATASET_KEY].training.lr
dst = Dataset(ds_train, transform=transforms['all_transforms'])
dsv = Dataset(ds_valid, transform=transforms['base_transforms'])
dlt = DataLoader(dst, shuffle=True, batch_size=batch_size)
dlv = DataLoader(dsv, batch_size=batch_size)

In [ ]:
device = torch.device('cuda:3')
unet = get_model(cfg)
unet.to(device)
optimizer = optim.Adam(unet.parameters(), lr=lr)
loss_function = nn.CrossEntropyLoss()

In [ ]:
training_stats = train_loop(
    model=unet,
    train_loader=dlt,
    val_loader=dlv,
    optimizer=optimizer,
    criterion=loss_function,
    device=device,
    cfg=cfg,
    log=True
)